Como ya sabemos, vamos a impelemntar dos algoritmos de Aprendizaje por refuerzo: SARSA y Q-learning.

En nuestra impelementacion obteniamos 18 observaciones continuas del estado y sabemos que para estos dos algoritmos necesitamos observaciones dsicretas para no tener una tabla de estados enorme y debemos filtrar esas observaciones:

- Discretizar valores, angulos y sensores
- Discretizar observacion completa 
- Implementar SARSA y Q-learning con esa discretizacion  

In [5]:
import numpy as np
def discretize_value(value, bins):
    #Devuelve el índice del intervalo en el que cae value.
    return int(np.digitize(value, bins)) #np.digitize devuelve el índice del intervalo al que pertenece value, dado un conjunto de bins

In [6]:
def discretize_sensor(sensor_value):
    """
    Discretiza sensores normalizados:
    0.0 = obstáculo pegado
    1.0 = libre
    Vamos a tener en cuenta 3 estados para cada sensor:
    0 = peligro
    1 = cerca
    2 = libre
    """
    if sensor_value < 0.20:
        return 0
    elif sensor_value < 0.60:
        return 1
    else:
        return 2

In [7]:
def discretize_angle(angle, n_bins=8):
    #Discretiza un ángulo en radianes en n_bins intervalos. El ángulo debe estar en [-pi, pi].
    angle = (angle + np.pi) % (2 * np.pi) - np.pi
    bin_size = 2 * np.pi / n_bins
    return int((angle + np.pi) // bin_size)

In [8]:
def discretize_obs(obs):
    """
    Convierte la observación continua del entorno en un estado discreto
    Observación original:
    [0  x_norm,
     1  y_norm,
     2  cos(theta),
     3  sin(theta),
     4  v_norm,
     5-12 sensores,
     13 distance_to_goal_norm,
     14 cos(angle_to_goal),
     15 sin(angle_to_goal),
     16 cos(orientation_error),
     17 sin(orientation_error)
    ]

    Estado discreto usado:
    (dist_bin, angle_goal_bin,
    orientation_error_bin, front_sensor_bin,
    left_sensor_bin, right_sensor_bin,
    back_sensor_bin, speed_bin)
    """

    # Extraemos variables principales
    v_norm = obs[4]
    front_sensor = obs[5]
    left_sensor = obs[8]
    right_sensor = obs[9]
    back_sensor = obs[10]
    distance_to_goal = obs[13]
    angle_to_goal = np.arctan2(obs[15], obs[14])
    orientation_error = np.arctan2(obs[17], obs[16])

    # Discretización
    dist_bin = discretize_value(distance_to_goal, bins=[0.08, 0.15, 0.30, 0.50, 0.75])
    angle_goal_bin = discretize_angle(angle_to_goal, n_bins=8)
    orientation_error_bin = discretize_angle(orientation_error, n_bins=8)

    front_bin = discretize_sensor(front_sensor)
    left_bin = discretize_sensor(left_sensor)
    right_bin = discretize_sensor(right_sensor)
    back_bin = discretize_sensor(back_sensor)
    speed_bin = discretize_value(v_norm, bins=[-0.5, -0.05, 0.05, 0.5])

    return (
        dist_bin,
        angle_goal_bin,
        orientation_error_bin,
        front_bin,
        left_bin,
        right_bin,
        back_bin,
        speed_bin,
    )

##  Q -LEARNING 

In [9]:
import random
import pickle
from collections import defaultdict

import numpy as np


class QLearningAgent:
    """
    Agente tabular Q-Learning.
    Usa estados discretizados. 
    """

    def __init__(
        self,
        n_actions,
        alpha=0.1,
        gamma=0.95,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.995,):
        self.n_actions = n_actions

        self.alpha = alpha
        self.gamma = gamma

        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        # Q-table:
        # clave: estado discreto, valor: array con valor Q para cada acción
        self.q_table = defaultdict(lambda: np.zeros(self.n_actions, dtype=np.float32))

    def select_action(self, state):
        """
        Política epsilon-greedy.
        """
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)

        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state, done):
        """
        Actualización Q-Learning:

        Q(s,a) <- Q(s,a) + alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]
        """
        current_q = self.q_table[state][action]

        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])

        self.q_table[state][action] += self.alpha * (target - current_q)

    def decay_epsilon(self):
        """
        Reduce epsilon para pasar poco a poco de explorar a explotar.
        """
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, path):
        """
        Guarda la Q-table en un archivo.
        """
        with open(path, "wb") as f:
            pickle.dump(dict(self.q_table), f)

    def load(self, path):
        """
        Carga una Q-table desde archivo.
        """
        with open(path, "rb") as f:
            data = pickle.load(f)

        self.q_table = defaultdict(lambda: np.zeros(self.n_actions, dtype=np.float32))
        self.q_table.update(data)

Entrenamiento q-learning

In [ ]:
import pandas as pd
from tqdm import tqdm

from envs.continuous_parking_env import ContinuousParkingEnv

def train_qlearning(
    episodes=3000,
    max_steps=300,
    alpha=0.1,
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.995,
):
    env = ContinuousParkingEnv(render_mode=None)

    agent = QLearningAgent(
        n_actions=env.action_space.n,
        alpha=alpha,
        gamma=gamma,
        epsilon=epsilon,
        epsilon_min=epsilon_min,
        epsilon_decay=epsilon_decay,
    )

    results = []

    for episode in tqdm(range(episodes), desc="Training Q-Learning"):
        obs, info = env.reset()
        state = discretize_obs(obs)

        total_reward = 0.0
        success = False
        collision = False

        for step in range(max_steps):
            action = agent.select_action(state)

            next_obs, reward, terminated, truncated, info = env.step(action)
            next_state = discretize_obs(next_obs)

            done = terminated or truncated

            agent.update(
                state=state,
                action=action,
                reward=reward,
                next_state=next_state,
                done=done,
            )

            state = next_state
            total_reward += reward

            if info["is_success"]:
                success = True

            if info["collision"]:
                collision = True

            if done:
                break

        agent.decay_epsilon()

        results.append({
            "episode": episode,
            "reward": total_reward,
            "success": int(success),
            "collision": int(collision),
            "steps": step + 1,
            "epsilon": agent.epsilon,
            "final_distance": info["distance_to_goal"],
        })

    env.close()

    df = pd.DataFrame(results)
    df.to_csv("results/qlearning_results.csv", index=False)
    agent.save("results/qlearning_qtable.pkl")

    print("Entrenamiento terminado.")
    print("Resultados guardados en results/qlearning_results.csv")
    print("Q-table guardada en results/qlearning_qtable.pkl")

    return agent, df


if __name__ == "__main__":
    train_qlearning()

